# KB Ablation: `--use-kb` vs `--no-use-kb`

Compares the effect of augmenting the rule-based ATT&CK tagger with retrieval from the cyber-anomaly Chroma KB.

Both runs share the same parser, windowing, chains, IsolationForest and GRU. The only difference is `tag_techniques_with_kb(use_kb=...)`.

Outputs compared:
- # of windows escalated above threshold
- distinct ATT&CK technique IDs surfaced (rule vs rule+kb)
- average hits/window
- per-window technique deltas

In [ ]:
import sys, os, json, shutil
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault('CHROMA_EMB_DEVICE', 'cpu')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
print('ROOT:', ROOT)

In [ ]:
from pipeline import run_pipeline

INPUT   = ROOT / 'data' / 'samples' / 'sample_lmd.csv'
DATASET = 'lmd'
THRESH  = 0.2
OUT_BASE = ROOT / 'results' / 'kb_ablation'
if OUT_BASE.exists():
    shutil.rmtree(OUT_BASE)
OUT_BASE.mkdir(parents=True)

print('Running NO-KB...')
res_nokb = run_pipeline(
    input_path=INPUT, dataset=DATASET,
    output_dir=OUT_BASE / 'no_kb',
    skip_judge=True, skip_detectors=True,
    threshold=THRESH, use_kb=False,
)
print('\nRunning WITH-KB...')
res_kb = run_pipeline(
    input_path=INPUT, dataset=DATASET,
    output_dir=OUT_BASE / 'with_kb',
    skip_judge=True, skip_detectors=True,
    threshold=THRESH, use_kb=True,
)
print('\nDone.')

## Aggregate metrics

In [ ]:
def load_windows(path: Path) -> list[dict]:
    return json.loads(path.read_text())

w_nokb = load_windows(OUT_BASE / 'no_kb' / 'windows_scored.json')
w_kb   = load_windows(OUT_BASE / 'with_kb' / 'windows_scored.json')

def summarise(name: str, windows: list[dict]) -> dict:
    escalated = [w for w in windows if w['detector_score'] >= THRESH]
    all_techs = set()
    rule_techs = set()
    kb_techs = set()
    hit_counts = []
    for w in windows:
        hits = w.get('attck_hits', [])
        hit_counts.append(len(hits))
        for h in hits:
            all_techs.add(h['technique'])
            if h.get('source', 'rule') == 'rule':
                rule_techs.add(h['technique'])
            else:
                kb_techs.add(h['technique'])
    return {
        'config': name,
        'windows': len(windows),
        'escalated': len(escalated),
        'distinct_techniques_total': len(all_techs),
        'distinct_techniques_rule': len(rule_techs),
        'distinct_techniques_kb_only': len(kb_techs - rule_techs),
        'avg_hits_per_window': round(sum(hit_counts) / max(len(hit_counts), 1), 2),
    }

summary = pd.DataFrame([summarise('no_kb', w_nokb), summarise('with_kb', w_kb)])
summary

## Per-window technique sets

In [ ]:
def techs_per_window(windows: list[dict]) -> list[dict]:
    rows = []
    for w in windows:
        rule = sorted({h['technique'] for h in w.get('attck_hits', []) if h.get('source','rule')=='rule'})
        kb   = sorted({h['technique'] for h in w.get('attck_hits', []) if h.get('source')=='kb'})
        rows.append({
            'window_start': str(w.get('window_start',''))[:19],
            'detector_score': round(w['detector_score'], 3),
            'rule_techs': ','.join(rule) or '-',
            'kb_techs':   ','.join(kb)   or '-',
        })
    return rows

df_kb = pd.DataFrame(techs_per_window(w_kb))
df_kb

## KB-only technique frequency (newly surfaced by retrieval)

In [ ]:
from collections import Counter
kb_only = Counter()
for w in w_kb:
    rule_set = {h['technique'] for h in w.get('attck_hits',[]) if h.get('source','rule')=='rule'}
    for h in w.get('attck_hits', []):
        if h.get('source') == 'kb' and h['technique'] not in rule_set:
            kb_only[(h['technique'], h['name'])] += 1
rows = [{'technique': t, 'name': n, 'windows_seen': c} for (t, n), c in kb_only.most_common()]
pd.DataFrame(rows)

## Sample evidence pack (peak window, with KB)

Verifies that KB candidates and the peak chain actually appear in the rendered evidence pack served to the SLM/Judge.

In [ ]:
ep = json.loads((OUT_BASE / 'with_kb' / 'evidence_packs.json').read_text(encoding='utf-8'))
if ep:
    peak = max(ep, key=lambda x: x['detector_score'])
    print(peak['evidence_pack'])
else:
    print('No high-risk windows in this sample.')